# Phase 10: Advanced Transformer Intent Model (DistilRoBERTa)
### Google Colab GPU Training & Golden Evaluation Pipeline

This notebook fine-tunes a pretrained **DistilRoBERTa** (`distilroberta-base`) model on the isolated AppleSupport intent training dataset and benchmarks performance against the human-verified **Golden Evaluation Set**.

**Key Constraints:**
- **Hardware Requirement:** Execution **MUST** run on a CUDA GPU (e.g. Tesla T4, V100, A100). CPU training is prohibited.
- **Strict Training Isolation:** Asserts $S_{\text{train}} \cap S_{\text{golden}} = \emptyset$.
- **Zero Data Leakage:** Only initial customer message text is used. No agent replies or future conversation turns.
- **Reproducibility:** Deterministic training with fixed seed (`42`).

In [ ]:
# Cell 1: Pre-Flight GPU Hardware Validation
import sys
import subprocess

print("=== Checking GPU Availability via nvidia-smi ===")
try:
    nvidia_smi = subprocess.check_output(["nvidia-smi"]).decode("utf-8")
    print(nvidia_smi)
except Exception as e:
    print(f"nvidia-smi failed: {e}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "CRITICAL ERROR: CUDA GPU is not available in this runtime!\n"
        "Please switch to a GPU runtime in Colab: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"\nSUCCESS: Connected to GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
print(f"PyTorch Version: {torch.__version__}, CUDA Version: {torch.version.cuda}")

In [ ]:
# Cell 2: Install Required Deep Learning Dependencies
!pip install -q --upgrade transformers datasets accelerate evaluate scikit-learn pandas scipy

In [ ]:
# Cell 3: Environment Setup, Repository Synchronization & Dataset Resolution
import os
import sys
import gzip
import base64
import shutil
import importlib
from pathlib import Path

REPO_NAME = "ai-customer-agent"
CURRENT_DIR = Path.cwd()

# 1. Ensure repository code directory structure is active
if not (CURRENT_DIR / "src").exists():
    if not Path(REPO_NAME).exists():
        print(f"Cloning {REPO_NAME} from GitHub...")
        !git clone https://github.com/mohanraj9342/ai-customer-agent.git
    os.chdir(REPO_NAME)

print(f"Active working directory: {os.getcwd()}")
Path("src/classification").mkdir(parents=True, exist_ok=True)
Path("models/distilroberta_intent_classifier").mkdir(parents=True, exist_ok=True)
TARGET_DATA_DIR = Path("data/processed/apple_support")
TARGET_DATA_DIR.mkdir(parents=True, exist_ok=True)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# 2. Synchronize / update Python helper modules
train_py = Path("src/classification/train_distilroberta.py")
eval_py = Path("src/classification/eval_transformer.py")

print("Deploying latest train_distilroberta.py and eval_transformer.py...")
train_py.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/608a3PbSHLf9SvmsOUs4AJhy/ZeNkzxqrR62EpkWyXJu7koKhwEDGmcSYABQNncjT+mKn8lfyu/JP2YJwDSdu1u3VnAoLunp2e6px8zDILgoG3yJ/kya9tyXuZZV9bVk67JyiotyrYrl019L5suS9bbg9nX/ndwIuVa3DRZ1c7rZiUbcVZWcnKzqcpqIS7LtVzCuwhPqIer+qfTq5ssEgArjtbrpbzerNd104nzqpNVJ4497pKDg3+VW/FykwH9Tsp2ejAR111T5p14eflOnFZAJ5crwJyKo7YF7lvRQdP7JN8UWVK2afaQlcvsfinDSNxLAJeChozc3ctFWbXJgRAXdZ4txTGQNB/LVrTU03Ir1k39vrwvO1mINYywhn+IP8BZbMqChgh0DG83msh5Wy8Jcip+zpZlkXWyFdcpdSL+73/+F54X9bKQlZjB639rDucowo5EiFTPq/UGiMpPnTj9lC83bflQdtupeFsBb9BRVwIj+abtapQ/MAvDXQH30BuKphXh3zpA/lskMiC+aWWBVIlUAfx8X0lZtGkjH0r58XvxSm4anKxcHNfVfAnjaafidQ2jRJkQ6yAHI5u6EoeHIq+rvJEdCDf7VFf1agsPzUJ2JBWa1Mkvsly878RP2TKrchjZVHALEDtuagA4rbqmXm/FEl5EVhSNhBmFWZAPEtimdSvK1T3hS6R71HTlPAN5X8qmBZYltE/FtWxAHuWvgKrlCF18pL7aGJbHB1nB1yZGpuflIhZZVdh5X8kug3nKsIOXPDfXEgT/kC03aiqP6xVMCNDPgVVZTD7WzbJARBAJ9ACrY1Lycm4kLm5oa0GO0oJgj9nqvlxs6k0rijJbVDWKHMQVgJ4ezJt6JdJ0vuk2jUxTGDYpSVZVdUdMtAcHuq1ZrLOmlfr9721d6edlvYAVvtCvdauf2m3LfeCK7MqV1D3od5AT/PtrXUmGW2fd+2V5r8Eu4ZU/dNs1aQu3H1XbWJzAwojFBUxIDKL7zw1OSyxuNqDthutqs4KZzlpRrXXTGoQCDfC/dcG02w9LmTVVosSmOwlBYYXI8nzTZPk2bXNQmZjafNuWsvTVJ5hrUBxoXWVA7RO3rhuZly3DggVYpnOilrZsleKDqMcJqkHayqXMsQfNERtRWBFd2q6XZacmEMxt4rOU3G/KZaF0PpVmTQHJzh/e8dH1aXpyfnZ2fvzu4uav6c356dU1M72ssx0k+DuoSznfpnpFp6U2QnY4Q8Z4waZIG2yIYgUUOFUfqmwlYdHhioLvM720EgC5oLYwJZg0jQ4OTk7PjoDr9Obq6PxNenz9MyAEqFRPwDjlqNbFkwytvxa0/6b7NCPIYWmw8Uzy9iEw9F++vTg5/R0dKCGC5Hy6b9/dXL67AfFfIV2a9PaJt0dqClqKsrHYpz8fXaT/cv32zTcy5XfgzCwYws0STCmqdmCl+/rtyelF+ubo9Sl15GJP7rNWWo6uT09PAObFM4t79G/pxemblzevoP3w2Y/mw09HN8ev0uvzf0eizy3CxenR1ZvzNy/Tq6Mb+iQnP9gRX749fnWNrabpl6Or1+8uEfr8LXx4mhzaT6fnL1+BdE+Pj/5Kn54eHhwcFHJO6w02a1CxFYgGZr6FXXvyF7Ydt2hXbmHfiWHP6+7YztzCY4yb0d3dlBY/mM8rCVazErBo6qrEfZ1IPisLMrtl8Yzehe6CfJHuvcRtzOxeNK+w3NAaI1lXC4DnFuZLFmFfP8IoYg3VHc7Eb/hhKkrqpYwFvgI1IcECohchQ5dA9Jl70zwCfjllnG/Bb1gCmo3YEFRyVhZisd54/hFI2soYTLkV6c+EAFLKYMMRx+9Ojsj/An/A4LMYHT9Qay84cUjlKitxN7/aVLiznDYNjmfOtIDOpjKUkh3umPXCEs0X/e2aLTNKolMWGZ1AapSfcrkG/5LauVvYYqDV4jTImsdZaL5RR5fbG6SHTMAGDOIH32q5BK8CJgJXDuxEDW73snoom7pCpysRQY8G7CDQi6xa2NLFtt6QM9ZsKh4eEzoGS31Pom2Ym8RSAacZbTdwzgIF4SEzO93dbxjf8dX5zfnx0cXUm1qkbuj9aY9fzcsNxHGWLVs5HHnPZW7AJygbdk1Ri53AQKzAiwUv2LiZsMdmzE8F3li9WCoZDTr5ag/elSgLsgDHN5dpXoOAQOOccbpfwsgFJjX0YNEWOB/Dpx58nq2z+xJcg+1OLAuicTtw9sAUylXdgLLeA2YDnBThOD4MEYTclWCGnkaJiyueiPDw6bMXjx8/j2LxTA0btb+EAAqNjBFJgFTt5AZTiGU2yrui765I4Kv7OoRCQVggfHNgcvainXED6Dz4bSCO26d3n5OR5sO7z4FDryctINZr8WBBgukDhg2wpU7VjKS6JU0Z9rPyt9C/SVBWYYBrUWmSIKtYymIqHkGMBf9/+ZPg3mJWJGfWH7WRw6z+z5XMgF0zQ7djsrqLXFOvQZWBB78WggKZovcB3k0bKkOJbip4Oim681PUDvFf5MszZ8odKuZTcMGTE8A9a8ycgTuSkgrDJIGX2PGm/pQ/thKlUJL+uE4HeJx2//Zo+j2I/Xs7bHvrBKKDpsmcTekCfFWr7dZHjMHIUl5AWM8XQsFlh3YIAmk/3uVgjNx2JvYERprYvQVFhuKCkaGkQl+IkWeKDXAiP4FZa4dG+Kxcyjd1d4Z6zJZ4HtwMhyDQpUWScwQUWTcVv1nqnwOlwt7KRHEglRGJ0M6BqzSIHSaVgZrDtH6E0cGENBIiCxha6AIR1HfiMNHZDQig3kv4h3wrI2KRv5f5h+me7Mb+4CRkRmK7CqPhGBUHjonXvT8YXYTQvhY1vC+ztfhYwswpJtDND8x4niU6AwK+B3jY9fIB9psVONrlBGIiuUDDmasMCBjej6Di/sox8stha8cx8ghu1Z/AyF/5iMGd+BM46i6N4C7J6/U2HA7V2qozWrm4F8LafVT00zXI2T8LZsH0KNZ1vRTvwdEBBAht66YAX9ban6XU8gb91G9EIxoAuc1Ges/BSfO8aM/vjV0PdsypH5EcPd6ahxHpgdtRVqHuI/kgt+hwexK06AQGEndoj5EElgzFPgnMmfUocFMCxmRZZWEQREnWdtu1DMFKGdm8SHSGSKhkVaGyVyoJRa4yZbm4AVM0Jn3Be2nrdasGk2BIqDZbGHrScihbyE9q+BDKZiswtYjtzZ76qqIa9dUft+ZthrYWdagNDUIsChzlDL6Q7X/+jHFwHHlakh8Mnv9CWhTH8uW0LbiDQ7clRMRYHEYGTnFwix9Q7HY04L9Y5h8jwRHboGRe7BP6o8JGdrEViJm6H8jIgUVBU2K3A1w0aGQ4uaNWCX2N8Qkg0KvrJYGsCutpsKrFUOWvcma21djZKaqiXqWUv53hzmo/tczcdjZYG1o/B2I5YReAeZ+iBNg+K9nG2OIOkZtRPGoFEXhkXgE2UvJSvkdfFvGIHYj1NCgHZaTs8CUfxXEtTF7J81u+gGSTRYwF5hTWC/DQ7ECwWSBG4OQfx/OIYCFtJoYh5brO37d9h4izJAxxn3XggOIK6EPZBIzK9mHiEXdKDPat7zWal1F8Zp9SmKwFSsKnbbM+3+C2UU7XJHZtcsA6Ymc6zd4L5mA5obu8aXGrfrWhhOFZBn4ZeTzgjdHuTDo4+ahrAWgXjftFboeq8hAtFZmQC9yPYsazGs5+Tw6jcgZseg39At/9pfTmrhxr2Ftt7pZ4yW43ZtEzciuZJeumjNqOcX1he2mt8rhLP1SZmf9qbYcZ3sw8OYYFVoJja9SgBskUXTMgDvRnZWEsgJMF6iXt8b+jTVdTOemsbnR5wC/7xR7sjSnYmGa1fnoNsLqOmsWG8wrOyMDS9gemG0P8x924TWd7/GsNohxqaxZ01K6+w6R5/CcoHAjTpUpuhC6mShbMDXpKJiKkf6eO6mFd5Zb0b2fCzrHNhhk/40NUtTvjh6XrrMBhzgJrRnqBa9dsKp6omZ8c8I3PzD7G/ZTLd1hbBOXAteVZBrWW2J0EL5f3qpluVyKkSpHZm27VQGKhN0MVGwM+b807sOHjKO5w7tUcmvCKldvumlolkgQjDJf3FGYAfTn1Sg6nP8UxzwboH0rT5dziYkj6ZUzrxJB+DYdxzrViHkirdK9XNeP1rFIaxl2KvIUe+/5jZLdHteb36/dAEczycLowbdVmxfFDO/N6tRDaaM6M9bQxjIKeGRPrxzF/Bl+dCuf+nuRWrAf16Qt4sf66qWIrAqH66zjAqNZpioX6NAWLs5zH4nHWLMBBoG3Vs/TgQFbg1+uU1A29gW/ypq4w4Yh/APvxh4+GAJkBbJ96ithu1sBMlJiOqUuLG/nQwFXS33LGGDvwRqXzUrhpq5Gp5UMjK/HgQjvtGalYGaeUPTD4fo/R6ozTx2Ojg7/+4Hg9AAr3QPEEK3Ab+ANTfQAoMRY+fswoPhRoSUlAClwRpNYewXI+JiyVLqdZQtugCA6amW+neTrICaIs03luU9FVldACVOsPl1/I/c6GnCRdHXLnKnsdRaM9kI/DHWl4zCeEk8OYx0fiSviIRGJ1MFLeigHv0YcvO8ZkpQtBC7wHdx6Y2rBC/BRryAjF7a8W6oEI2j1TL0N1UCBEjw3tS2GXkF2C5EE7OyWPPrZrymAbEHxRkTEsTNjXQo2UfSrbmRPDZnkOcP7ZBDZZeDAEyVjYVZY3dTo/BIT54RgokIc9MltI3IwBFmwwBuYQsjzQoYXZ037wLIuvpKfB95FUM/KbX2jRYwtULBJCQ+T7AIEemYHRDX1Ah2sD67Q54J+1xf7HxB6vMu4eW+2TLewcZS6uy0WV4ckZcV61a3VUI3xXlZjeB4fgZ87yU5YGNiSOJiIdFHrZXhsk+t+T1QdoC9Ejh+7ZDxKU9E3rD85GTknRdQ0hVotEWAe4hycicL4GB24WRhlbYwL4NfQ0XSdlGMLkZVzHveThs8tbLsheUkvSahGFA9fZ7BhRAsODrbgDcTFZtMowjLzrm3S/hmSFFlCkHPak4Exr4IW3AO69O3CwmelCF/tVNoIGLPsyjkIqvQNDPBbPHCw0ddwDR/EAyw8ODM8AEM8z1IOxgw3uEDGWvMeMD7s3WZfKqhiW1th6pRA4OdAAZtXJAV40EgTUpGULwB08Dem1oO0pF5WW5arEct2hxxa53WnbyTWO8oenLi7ERTh5XvYp4ANVQBO5qmD3CnSdTCnnfH34Z6HOs+jKSICNAaYIcQnSWbsvl6zNSrtldFxhODzdkVLhSZvNpRfLY4JMLraC9wDzCr1X8iMe+nOjeecrhsSNqeYEHrZmfpQ9HxL5DGi9cAJDLjW1Xp9fpNmHdymPCQFn2wzfjIPWwFf16UN+qTdw0FabtQj5L2cwHSGrZlpaPenyguQvMwwXQ/A4QkouOiFTJJ48cXQ0AiVlLbRT5Ha9d2Qe4J2T6nLPKDmz5TL/NYQZ8I6saxe643s82pUJPriygxTVLgbBsDmxsMLBa13Cg8fGFot67hxBcZMtulBbFimyiVb5w1Q8UOr7QwwPGLlq/pOykys8YAVj/qCG+dnmYNE0KCKDLQK8aNuLGk6n4qfZICDyo7sZ/Rt78px5Pca9zJYKrmfu+rAwpH8OiIqbHSI6BTLrhpmknuc46707gCOx0Mx/9ULLXtkya7DK08uPOse8Va7U5g94pHzyTxcXgB79HSkXOkXkGhP4eNBFvFzW9+Du0FrEzH9MoSs8JS/mI4cR3C6TBeHSOo79L2au0Ac3a/nHRFyjBeKD4mjlTd7LnNNuRySTPSDTzkHtlSFgE2ldrXJt2nmK3CWXkOni8A69DQPUy8cxnJN26AGrkfyTGomR6Gt1IpyzHOrF93jUxg7bGihVjn4V7pDebLubN57OTG2uA2DHEh+BjbrIMRrNfQS6DUG8HAd91RkR+DpMjgRmKlXVxxzr8Us/FsGmvHahYHnIQdjtQ33Zf/ta19BJVE7FWNIx0BW1UafGCAHP9IBVXa3xyJI6B59U9cdQH4VPNl0egctSo7mFSMUdKFYdnON/QEJXJsa6ovhXBzx7tMvtwCokotJWs0NhI/f0EkVG9Rpmx408eL2Zuw50sjiGrS/AwzN5zRngTTef/BhEeF5ybjdAhE2KzWodanSIqTHXU8C4Z8/82iAbe/fShUZShUDl6pBXr/exFJaXPZod2uwip5V8rXba9p5f0st/2j9b9A01s31HZDGYbOR72AkgzHRdUtit8WinY+L8XSBbZLibE5C6cGIvm+DdE3sIebQc0+bleptgWIfXgPQFj3rewZgOBoUt8+wemzCNzqGHQdveUw+cLcJRe+ckMWcCyFzpcMLJMIpUekpbXb1biPlmuXRO7Ijw8Icf6ZAL06XVKQuKjAeljSHPXb0EYbuaqusbfrC0s6bxhXqGyk2xA9DOgnUXuJV4yjWmJSWPFOe3gWkk/kKVpuMAu8NzKXxLpf3gIvlffEyr5yphWKeLJivceKqfBjU8zMxT3Ot95r9GBzuTpSp/mK83YZTQzR5ddsW7aHxenpbjaNoMU1N4igWG4ufYCNmHdSYY0UATCVedrpqJW63lt+Udn5hHx9bt4e7Apu5tf022Dcdx+VgLbm76cxTd2SoWXwD7hS6AHR5OqG7gqi+s3h+EOV2mTmKplYyeKn52dc0ZoHr8yrNjeKRuIxU9TXpAhE8QtQqD5NLHGMrVReJbfanJlaqLEKRkaKstG5H2qDglanKFbmLUQscOO7qeRFkP8D75oYlNwjQWKRfE993dslZhvJcvJ1cdPTY5yXVsn5vYzbr+0UztztC6YUa+nKtrbrpe07/+9uUuVXHNm9rYHo/JO5Xe3C2efIW9927Yhd/Sn9Goa7woObmQDxAEXMqGPC0sUuIU4bVQ2Bx+FEU5h1GC17MVXanjXrpima4tzliS8rM5t4aIFPU6K3j00l3kmFFE6u2jrsrm6Ndb5tAkzQjnzqdAOQJ1Ck+RtOYVwnEXaCae+vUUkDPEkhvZR0DQwFwqTekkq9bhXmmwL6lb4tELafxj9XhE3vA0jByDqiYPPVCWD+8H42kVeFpuxUOZ8VXbgsrNGNqtYO6pip4tt23pHU21nut4OQnUCVSrU9kWLb6huVSS37FXREm7WYW92tUfLxfFLICppxEYp6DC1zv0CJ+4tMWLqC8jpTFH5h7xcYZb2om5TQyrnuUrwuf+3gPLZM8yHhHlnp3HLjk9nVM+pNI7Lobbs9p9Uf/SGL06ykgRM5iOatDPCz19K9REwxdw6j5KvHFX6MsPxn3gSvyMig0EOTLjBgM9TCDyyfOG6RitKwTbFy+l4u726Z2tfq9T5H7GHs6tJunoeb12nBqCRr8GrU0Y3U6eT+9up9PJoY9gvZJbb65/C7QeczmFwcBPiWJcY1q1zArizZY6ZaAX0WePoHFvHDYNwJ2THTNaq3dwJuqggVTu3JomKfZeaDERw/ZDPM8EloxMom2PxF/EIZd8LTN2FoZrL8nWEOsW/nGnMfXV04saXIyoJQUQU7uiOKCIRiAHa23qLMpRozCYMNs0VPQeTsoiNqj8ugMNJanYwx8NoJtOZpmNwL/fgN1TWg6mB8166wqBG+6iXebaD/4HCTKV7PITYyJ0fj3EvSTllkF+V27GLafgKWhkQWk6HsQeXiJTH5XJVMktey/FvcRGAUBKvwChD4qia+hhaufaRcQLKugfqkQzQO6us/NEQ8NwllVt0LidPROgvkY7EdlLHcVqdmNRtX6IgQX7IYr1nXcwaQH2oo+yar/uxR0w7B0y8BA/9yrI6nq1nSfrcnuXGX3ndzifJpHr+9l98SrkfDWSunBZY19liX6yw9vAg3EwrK1EN9VaTEAbmlFT1+XrhPgDI1Sb4Vvi+jdHkjd46Xyd5cpHo0YsARkAXTK6pC9hIdu8KdeUZ3GOjaNLqJVn9Ay5ropwB0lWUOmJKIfBZEKJ0An+hEOM54EycH1ng7sCeymwcu8gYW8O7KXBEdMEjz0MadjLBHtpoBQm6vceRqiYn5bYS4SM7YSM7ZCEvauwnxFO3MeCzphQtnTADN1n2EuF8voTSvLvo2TvPOylpgsAEyoIKIKkz0OS3r2I/dLKPk30Oec9PNo88F5qVGPYRwfvV3hXdxUhV8eU2q2o1ucf79S/u3KftWV+TKfzQrICM/3l/M3Z21jwNjgLHoVZm+MmGbXi9hGD0onz9k48CvlpCk+gx222gBelaar063PF/ja648F/VLPf+5/WaSYo/N8D862A+5ti/+Dm1/QPjPm0/gjWdF6ajntzUZJtkc13j9Y1zE2wkctNI7c1ULSJaRvc1vDhbGPcy+zisSqGse/xyLFqhhmrNrLC8/d+sc6W5/j7WLnOq84x2I6CnZNSZ25G8up0H4W+Du5ufKfnX9KEmJ/o+pY7PD1p8uJJv+7aKnlzykSrzOnXFLB2HoDYez5h76WdPefevyRjm2/L/GqVHhhuMna47pFIXiGOECIfLuFDkV97QtLWJw2Fby5Eutz0ipFjJw5kMTZic8jA8GGEdIlmRbSbFQRbfLZqpXPmCvt24NDf+dZyMpn0bJqb3fQWMhcNvNLClUreCCDjmbp5cKTihKkJdle33+vg4fu7afJi/rmH8hq99cnZoYeiXfhxFH2mx2AhiuNE7+voUvv9U6cjEwzsw7wil3/qs8hxgEU7OCjxegX/+BjlqtIUd840VWlP3kYP/h8gFq4tAVMAAA==")))
eval_py.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/51YW2/bOBZ+168gtCggY2RlOtiHnQBaIG3sIos0CWx3XoKAoCXK4UQStSSVNtvNf5/DmyT6kg3WKBqJPJfvXHmoOI4jKYqzoiZSsooVRDHentFnUmMlSCsrLhoqsu4lyt/1ixbA2hspiLQl+sybjgh4f6bokpFdy6ViBVq0O9ZSBNLREh7mm76lJdqMGtFXXtJaZlF0zUkpUaWplKG6ZCCiXvFPi9WGoEbTGVWKP9GW/Qd4K8EbVDL5lCJq4VCJAJB6pNFj35B2/kwFWAvCvvC6pC2aoF5TlRp5hYEOnCUTtFD1CyI7wlqp0N0jkRT9jrbwpwZgMkq+kj+5YOoFfXJrKdos51eXS/QLuoZXItD6j6/TRb5jxhUrugMlElTPsiiGeEQGPsZVr3pBMUas6bhQgKnlymCUUeTXxA4wSurf/5S89c8134GTd1ZcR9RjzbZe1h282g310gGRX79oX1JwcKEGDR24gkgE/7rSQYOEycKEybY9q0u8M87EdHAmllShAQ45QXFSKmQga3Fp4i34lgpFvLQkQvC7XCwvvl1v8Jfb68vFDf68/iMN1m+/be6+bfDl1cqu+2yY5jYGEA4WYLF0O6pwTba0xg3ptH9kGs2iSHsU0iv3rs2A7tqsJRi3pIFgAZVX/ulivbi+ulms8b/WtzfAFZdEkbNO8ALiTcszEF1TLPtOWxS+YdYq2qqpoyBJ+lrJTIc4HpRsVhc36+Xt6utihVeLNSz9X+oCH5/WGkUlrWwkdUFiU5DYlGBi/gdB4hxJJdB/TY4BjMNYzND8n0j1oP/e5NuQdPfAaF4fHs5NHKAcdPlPq38SOfSdst2jkulY+rZyG6qINn5sBJkuLC2xwZ2FpdGNmGdmk1UISszRZPQHOEUmMwtF/yAdoe6XrKY3XC1535YLIbhIqng5ArQNyfYMLl6MxErTIqLO0U8r/DWGRNEilXgZ5bvcBrbicVi0VTpaLYda7RU3XXLJxZr+u6dtQT8HBZQamo13ji2BHwXtFLoyMgx8Xduwum/mqm8Va6i1ML572WhUttFOwTQ9NMQtRboxkrrWEeImQ3S3DaJlHJPFM2sRaLQeKOkzKyhExJid2dckLvqSxDoidlm/Zkxi8kwYVGZNkxmC84GiuOj62IbPlmfG2oonJm90YwtPhXP0Qcapi7DlGgny0F2ZZsCdoKYN0TKB7Ewcp4veUY02Abw2OHfAIdaqQHvqFi0Ky5S/I6hvwsoUT7xYKxdKAWq11g52uX+GYpv3vkxccXtyUzk5+vnqa2KQcaQkvjOQyDvaJgNVimIBVgJyrv2Rx72q5v+AuEOeVSPnnjqNIdN5k1QOOljYi9Z6Jihxz+XakTunsYaCO300/46HkzlxVTa2e9fRzvc6jm38Ax/WeIy/TrSzsLvD8aCbWihy6GFfaEthBqIOqp2GOio0IgIBRg1Rgv1ApBBcwllb19ZqObStbdC2jqCc7cE/COL2VFMbI7h1efnu8B2omwZRH46JzTSd9D9fXVitDxhQ7odA4x3ke05oma/puApe82cjbCsi8bPEtZulILZ+lprwvVq9f0MXZTkZ2Uxvhe73ROH0MfhZu2dRxhRtAmeN4O89wId7kBBiN0j1PAAojJzRFWZMAHcAzywNGSCtBStkwMNhUIV0wH7P+HHkC0wL5uIx7T2v7rCHZfCGmuikwXEwLsR7gXOGx+GcnkyG+1k8iehodgB3L3b2HmGqZm7jj5Yfh4H8yDCNki3VJ5MLqDshhM/UvUAbLxxLo9ETwDsmn54BxYR1kpaHPrR2BcwnI3FSkDsrdcpP62Z8qqDu1MPDWPQ6ua0o6J5YFlzfZSDH9+EcZjm0CybNgQ7NKfG8UCWgbGZGgLj6ODersZHoKMLuoHD1UZvqNu9HnoeADhzpwOWhk41HvAXemRMuI37gtv4bdKTo1+zX2Z4pA7G3xYwQQBiItl6+t5oPK9tkbTgwVx8hfYUe8hIThkTbPkvR3/dq3LAeSbNDAca8UxIMwONK0RydYn2NDir63d1Ue8FSTI/mUZQ7is1dFMOlFNJJn4f+eprdQE+QHYERyPCbRT1wDQQXYtc3gOPO7MAMIwvBOj3v5LG7nNPTHwF8H3C3ebjCZ24mtJoyUpYallGRxPO5aWVzGNJjPYZVBAowP3JReUuEvTPOC/l8RMZ4IX1TxtCF5mYEO5QTThlvyuK9mrtWckTQqXvibBpPJ3kaRRfYBvxrQ3rDWxdEfwkGI1jxmbcV2yU1faZ17neubpa3KTJTjsrjDwmRhb5SzCS6/2BJ9Vkxkw/oQ2KfYEKGQVJKsoMXF0INBFIlRDXMzAeTobmeH7ugatZs78bnLv5l5bmOfqCwrG4LAm55cYpY+Zv5TADch58MPEzzMc1FBgj/9zeIZChbgza3dg6Lg735aPmwORiUD0/jpoeb+we7dTCYAcq3h+qpSakJUBbOpE4k5GQwuBpKveh4ZwGVTjzd9psniE9iX2S+ET1cKczcivmTebVs4+DqBcDo+v0do6uZUsu+6ZKf8RhoaKahWfHoENgbX14hpSH0LXhX5b8duXquyTN0KBp+CJ14V3F3EfS4dZXBQeW/IaE8RzHGuuYwji1uW4DRX0g42newFQAA")))
if "src.classification.train_distilroberta" in sys.modules:
    importlib.reload(sys.modules["src.classification.train_distilroberta"])
if "src.classification.eval_transformer" in sys.modules:
    importlib.reload(sys.modules["src.classification.eval_transformer"])
print("SUCCESS: Model pipeline modules deployed and reloaded.")

# 3. Resolve uploaded dataset files
TRAIN_CSV = TARGET_DATA_DIR / "apple_support_intent_training_candidates.csv"
GOLDEN_CSV = TARGET_DATA_DIR / "apple_support_intent_golden_set.csv"
BASELINES_JSON = TARGET_DATA_DIR / "apple_support_intent_evaluation_results.json"

search_roots = [Path("/content"), Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive")]

# Scan for any uploaded apple_support folder anywhere in /content
print("Scanning for uploaded AppleSupport datasets in /content...")
found_folders = []
for root in search_roots:
    if root.exists():
        for p in root.glob("**/*"):
            if p.is_dir() and "apple" in p.name.lower() and "support" in p.name.lower():
                if p.resolve() != TARGET_DATA_DIR.resolve():
                    found_folders.append(p)

for folder in set(found_folders):
    print(f"Found uploaded dataset folder at: {folder}")
    for item in folder.iterdir():
        if item.is_file():
            dest = TARGET_DATA_DIR / item.name
            if not dest.exists() or dest.stat().st_size != item.stat().st_size:
                shutil.copy(item, dest)
                print(f"  Copied {item.name} ({item.stat().st_size / (1024*1024):.2f} MB) -> {dest}")

# Also search directly by file names across search roots if needed
if not TRAIN_CSV.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_training_candidates.csv") if p.resolve() != TRAIN_CSV.resolve()]
            if matches:
                shutil.copy(matches[0], TRAIN_CSV)
                print(f"Copied training candidates from {matches[0]}")
                break

if not GOLDEN_CSV.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_golden_set.csv") if p.resolve() != GOLDEN_CSV.resolve()]
            if matches:
                shutil.copy(matches[0], GOLDEN_CSV)
                print(f"Copied golden set from {matches[0]}")
                break

if not BASELINES_JSON.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_evaluation_results.json") if p.resolve() != BASELINES_JSON.resolve()]
            if matches:
                shutil.copy(matches[0], BASELINES_JSON)
                print(f"Copied baselines json from {matches[0]}")
                break
    if not BASELINES_JSON.exists():
        BASELINES_JSON.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/+1cW4/jthV+31/BzkPRIp4J75Q2KJobgiyQBEUT9KUNBI5Nj9WVRFUXO06w/72HvowtjSnbM5Z3vWvPxrFNivx0eG7k+aA/XiF0k8e5SeLMRFNTlLHNbl6jG3KHbwausdK/2cym8+1Gum4sdDayaVQaM4LfOV38aKY6qXUFfaMqTk1Z6TRfXIWpvMXhLRG/YPmakNeE3XHFCMefYfwar4ZM7cgkJfT/A7657/q/toireXSvywXKx6Z15yjTqfv15sdVV/RNossSfb2+YLDufgwygaUK2Ray9ghlnsSVu/rBJiMDI8J4m46VrXQSrZoKM7TFyN0UEcFjl2FiSzOKZrZIRtFqYPi+3Vk8drYgfp0kUWqqIh6WW0KARj0c1oUezuFXfIdDzAabtlQPCxvlMGq8Wj7oggP6pAv0gAnWQ4RP2sdk2UYE2Wqbmfhh4mB3zvDYqzkJ29VlM8+q8d2jEHJTRHFWmazyysHW0Aj/N2WzzV3dRDjYbtvG1WgYk9sS1sPsaCrrPLeFUwFxhx8b3m3dk87zyBZgHsU0HpooLsvanA8VxR5Y97qqTDGPcjszxRnxSB+eOAE7fYhyPU9hbc+HCMx8N6KhTfNEg6pFY3Bt93r49oygiBdUlplhFU+dN8xMBY7jjLCUB9XILHR7oovRTBdnVO/AA2hsdFUXgMjOosqecd185gbeHPxWOYnzHLT8fIBCD57Sjiu3VFGdjyDinA8Qox5EdfY2s7MsstVkr0OCqEGpCLDkgkhCw934SBc+IoUMMKQeTFEVEBz4VpT7/Ho73noQLSMn0tN9qw7RkhDCmQwFJlxK5pV62PwjHatAIIUhASAiQmHFFPHdpvDFr3VMPugOILUhMhCKc0yk5B130LGArTvgoeQyUCFRgkPeERx2B0/yBvCc49phjVINacNvzawh0ffLjPPf26O3conGzDvjeqNHM8Q2m1rRrtG4I/C02nfEgEaPtj9uirfpGxttLTfVFHXLZTQam9b72PJrww5WYt8W8fZnhBr2+sF8FVvffh1cGHiKLxq9vGT0hF00enLJ6NUlgw8uWm8u2uOElwye0YtWHL6N/lX70yaVKxOX3SRmanxnYel9/FDbuozSOqni1XlRO3tdHM+5LUkjk8nsYjt0883iZO52cTKHNkd+6KfPv/oCPZ7UodLkuoBPyRxNY41clhmPTDY0n5sS8t3lNTrTybyMy7ub3VuJCu7hvl6cKE7LqJyneWVTH1zKW9lgAZn1IgMetHLXrf0JJ2r3sQvktyMNSeooLpdSWyD2za08U+OOqXdv0GEr4DLbxUnnFLJk34ytfP8FUy5UwWXd9WLgnbOJU01WGNgblaB0IM6p8YoTe+ZT/vnI7vnqxERO+cA0qrNoTjmBfVeU2bicH6stwj+dgg1wxyZuY9hDXZoyWhtWY1fRxDIzporiRWmCYEGaEq/Mb4vj+y+/yvPE/LzcTDojRolGWQ1mjvSwAoWJf9fD+D81xmOWDRB4kweL8sLeJybVJXIuwF0Ft6eL1CwGMAn6OqlNZWFLNHDD5Rq0IYEPpUFVoe81mqPMIuhexNnIuC6wYzSFm8WM9F1zgwXqNIKlNaONM2ttutp7xJUf2nEQsmmEvXDxEGc7+lQ2j1Zzgj9o74ybQl5ccAisndAa7e8Gh87StR19Og9+9jwdW+IDpvHF45tJnWpXFJrGZgbju6jjhHzzxhmJzYZxokG50MJp3S7RIHeQ5Gpaa7VC2ycBA1QOC2MyNIb3380Ags4IrY4g0KjQMbTYJLEzuAtoQ0uxoVlcTWwNY6MSGmDGkU3jDFQVrcLQ7pjls7FQCuWxse/NHDXs7A59r0coBXSL4wrEUGqzalIukGv01ixaTGJzd/qExhpkMUATuCaukAsecKs2RWVlC7AoO9LzxZWgdWBUw4nO4jJF92YSr350A8Yl2NwMlZnOYdQ79JNdz/43lDjh2jGKs6FNzRdwDfSGf4DQZhDfC/O/OgaTgLlscoe+hYgJ9+HWZoT+jGZOZpVFJUh0CBY/BBm/QQ+mQuM6SeDiMUTav6MfNAg1h94lcoauGRtQgf90NfarsTeN/d44s8pQPoEAB5kkWp/moZFO9QMEGV1DgopGdeEsGhIOHRcr09nSrG3tQ5Dk5XVljjNpgqVUgu+06R/nYJZTV3A3LghWzny+JEQEIgAQFJMA7M3MUPyPCUTHtc8Bm3L2uLhgBmpjnImtXVWiy6pEFCNwQ4C1fLRpJ5ciA3vKwCkUyJ16lsh9vNrO1XaatuOCGgJvDVZTOg1xfJC1PQ11BRpW2HwSD1vx8S/bulf+dal5Lpfd6N5Yx0kNCpvbsrpdCnTbml5tb1lXortJ7ENcVvEQbuIBNgVlc3/V4q/88t3tm2+/Q5+hH1ZXoX9urnoeiYVLFaoLJLGIkIk9JBappOgisUhJqY/EIhkJ95NYZLBdxtpJYmnhbJNYRCh5rywWScUzCqBKhpThx3fWM62F+KvIzA+SSyJYsHmXPdJcgoAyQUNOiJJcYemDK1qS86OXoZKKcxJIzKUgtFdOTOjD20RLO6StAipxIEMhBSWK902Y8Ra4qWr++RHTALYbfPPeP52mNSHx3UNnv8Y9dPY8MfdGtl4+9Ep0aAkW26YieqTm8CZa5YXrR+t1GSeh6gRKeDE1X0GHAjT8jgj6pPIs6RYb9oXXyXVIFKuQyc17j0SfplzAO73c2sSLeD4u1POABQqHUilOj6f5uIyFSYaZVAQDVq/XlpQETCkRwC6KhR1e2+UxjArKZMAJ57xHho9LhiCckRCgcR74Q6RXTE+WA5Ij6rpjxoVSV27PR8vtER9kzfHAWjVpXiWPKmse00q767stGKeo9hJxDL6jvrYpGacQ/Z7q/ekWZo8ceii0k87FflFlvQe9OeqrPB34984ukS/Roh50/kUsqfet8yfUanUKye+hI3a28qOoOt2ilme3WNLbwhB65fW8iNfD/HwJ8fERe7rIKELIk3N7OuajfXB7RMdyKti79sHvCTumDK8Enw+V4LN/N9gqqsHWnO6tYLoj4YCcqoi5F6MP5zPLjMcUTfEdoQ0ae6910wZ/9cr9uXJ/Tsb9OeDUaFsTBaPqAC/ARHhCL9CJ0IfymabZdcS20wsoJp851d6jwJ2uQHJ5pQZdBDWo4yD3ib5ycoBVEcbYqazKD84H8EyxjmMcnNWgCA8/Hb5QnBkNEpqmh9CEFp3Rz//68ZnsoIBiJi+RHcQDvo8dJDu5QZRwHzdIhPKAB9xIwfc94KaF8gk3SDzGwH64QaL5ks8hCjUqq5z1/vyboPXyVRl5R1lREEU5YSpkxFUW+6QNscbL+8yDbRqTe+9CTyUjQgUhV66o2ydvqIOWJVqIOzSkb6YQ9ZbJSdD86yg1U4Ibgj0DWcjPDjmCHtQnH6iDvtLFAPqEOT8dBtNJrjpoTU9C8eGMNv7JZ3B8uGRKBpCthxyShl4pPq3Xyw2Gv+xRPtvEFbKDgXwAS0ZiTEJClu9ebYH0g2MZYAEuKQhFJ0cGC1gQzGDMQPTJ73FP2MEhFYQEYHpe7H4ZPQ3EVEJPFoQyCMHtXgk+V4LPBRB8gv6eaNM/waddfsXnIviQ59Xc6ekYCy+padOzsx3o6Srw7H0/surK8LkyfPpm+HTrGP2wGT64m+GjrgyfT5jh00Gz4UReH91zSfSejgkVDWQf9J6gi1F0Jfd8JOQeLuQh5B5G8Psl9zic5yH38GdP1HE4sLP+qGh4JfdcyT0fALmHSn6AE8BhEL5Hbg+V/FzUHinIWZkIgRBXas9FUHs6zzxby8ooZoeYFWXyVGZ1LOfGQXyuUR3FI6KhVOdly1H58ZN7Xrn/3v0fW1zRR2xrAAA=")))
        print("Unpacked Phase 9 baseline evaluation results.")

assert TRAIN_CSV.exists(), f"Training candidates CSV missing at {TRAIN_CSV}"
assert GOLDEN_CSV.exists(), f"Golden evaluation set CSV missing at {GOLDEN_CSV}"

# 4. Load datasets and verify strict mathematical isolation
import pandas as pd
from src.classification.build_golden_evaluation_set import verify_training_isolation

df_train = pd.read_csv(TRAIN_CSV)
df_golden = pd.read_csv(GOLDEN_CSV)

verify_training_isolation(df_train, df_golden)
print(f"SUCCESS: Loaded {len(df_train)} training candidates and {len(df_golden)} golden evaluation records.")
print(f"Training dataset size: {TRAIN_CSV.stat().st_size / (1024*1024):.2f} MB")
print(f"Golden dataset size: {GOLDEN_CSV.stat().st_size / 1024:.2f} KB")
print("Strict training isolation verified: S_train ∩ S_golden = ∅")

In [ ]:
# Cell 4: Fine-Tune DistilRoBERTa on GPU
import importlib
import src.classification.train_distilroberta as train_module
importlib.reload(train_module)

OUTPUT_DIR = "models/distilroberta_intent_classifier"

print("Starting GPU Fine-Tuning Pipeline for DistilRoBERTa on Tesla T4...")
model, tokenizer, metadata = train_module.train_distilroberta(
    train_csv_path=TRAIN_CSV,
    golden_csv_path=GOLDEN_CSV,
    output_dir=OUTPUT_DIR,
    model_name="distilroberta-base",
    epochs=3,
    batch_size=32,
    learning_rate=3e-5,
    max_length=128,
    seed=42,
)

print("")
print("Fine-Tuning Complete!")
print(f"Model artifacts saved to: {OUTPUT_DIR}")

In [ ]:
# Cell 5: Comprehensive Evaluation on Golden Set & Comparison with Baselines
import json
from src.classification.train_distilroberta import evaluate_transformer_on_golden_set, get_label_mappings
from src.classification.build_golden_evaluation_set import load_golden_evaluation_set
from src.classification.eval_transformer import compare_with_phase9_baselines

golden_df = load_golden_evaluation_set(GOLDEN_CSV)
_, id2label = get_label_mappings()

eval_results = evaluate_transformer_on_golden_set(
    model=model,
    tokenizer=tokenizer,
    golden_df=golden_df,
    id2label=id2label,
    max_length=128,
)

# Compare with Phase 9 baselines
BASELINES_JSON = "data/processed/apple_support/apple_support_intent_evaluation_results.json"
comparison = compare_with_phase9_baselines(eval_results, BASELINES_JSON)

# Save comprehensive results JSON
RESULTS_JSON = "data/processed/apple_support/apple_support_distilroberta_evaluation_results.json"
with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump({"evaluation": eval_results, "comparison": comparison}, f, indent=2)

print(f"Evaluation results saved to: {RESULTS_JSON}")

# Print Comparison Table
print("\n=== Comparative Evaluation on Golden Set (155 Closed-World Records) ===")
print(f"{'Model':32s} | {'Accuracy':<10s} | {'Macro-F1':<10s} | {'Weighted-F1':<12s}")
print("-" * 70)
for m_key, m_info in comparison["models"].items():
    m_name = m_info["name"]
    met = m_info["metrics"]
    acc = met.get('accuracy', 0.0)
    mf1 = met.get('macro_f1', 0.0)
    wf1 = met.get('weighted_f1', 0.0)
    print(f"{m_name:32s} | {acc:<10.4f} | {mf1:<10.4f} | {wf1:<12.4f}")

# Print Per-Intent Deltas
print("\n=== Per-Intent F1 Comparison vs. Logistic Regression (Phase 9 Best) ===")
print(f"{'Intent':25s} | {'DistilRoBERTa':<14s} | {'LogReg':<10s} | {'Delta F1':<10s}")
print("-" * 65)
for intent, d in sorted(comparison["per_intent_deltas_vs_logistic_regression"].items()):
    sign = "+" if d['delta_f1'] >= 0 else ""
    print(f"{intent:25s} | {d['distilroberta_f1']:<14.4f} | {d['logistic_regression_f1']:<10.4f} | {sign}{d['delta_f1']:<10.4f}")

# Print Slice Performance
print("\n=== DistilRoBERTa Performance Across 8 Difficulty Slices ===")
print(f"{'Difficulty Slice':26s} | {'Accuracy':<10s} | {'Correct/Total':<15s}")
print("-" * 55)
for s_name, s_info in sorted(eval_results["slice_level_metrics"].items()):
    if "accuracy" in s_info:
        print(f"{s_name:26s} | {s_info['accuracy']:<10.4f} | {s_info['correct']}/{s_info['total']}")
    else:
        print(f"{s_name:26s} | {'N/A':<10s} | -/{s_info['total']} (ambiguous)")

# Print Ambiguous Cases Diagnostic
print("\n=== Ambiguous Cases Diagnostic (3 needs_review records) ===")
for amb in eval_results["ambiguous_cases_analysis"]:
    top_s = ", ".join(f"{t['intent']} ({t['confidence']:.3f})" for t in amb['top_predictions'])
    print(f"Tweet {amb['tweet_id']}:")
    print(f"  Text:            {amb['text'][:85]}...")
    print(f"  Top Prediction:  {amb['predicted_intent']} (confidence: {amb['confidence']:.4f}, margin: {amb['confidence_margin']:.4f})")
    print(f"  Top Predictions: {top_s}")
    print(f"  Notes:           {amb['human_reviewer_notes']}")

In [ ]:
# Cell 6: Package & Export Model Artifacts
import shutil

archive_name = "distilroberta_intent_classifier_artifacts"
print(f"Creating archive: {archive_name}.zip...")
shutil.make_archive(archive_name, 'zip', OUTPUT_DIR)
print(f"SUCCESS: Archive created at {archive_name}.zip")

try:
    from google.colab import files
    print("Downloading artifacts to local machine...")
    files.download(f"{archive_name}.zip")
    files.download(RESULTS_JSON)
except Exception as e:
    print(f"Note: If running in VS Code Colab extension, files are already in workspace: {OUTPUT_DIR}")